<a href="https://colab.research.google.com/github/dewshishir/new-practice/blob/main/trafffic_chatgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

monowarislamshishir_vechicle_path = kagglehub.dataset_download('monowarislamshishir/vechicle')
monowarislamshishir_traffic_f_path = kagglehub.dataset_download('monowarislamshishir/traffic-f')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

# Check dataset structure
print("Dataset Structure:")
for dirname, _, filenames in os.walk('/kaggle/input/vechicle'):
    print(dirname)

# Count images
train_imgs = len(os.listdir('/kaggle/input/vechicle/train/images'))
valid_imgs = len(os.listdir('/kaggle/input/vechicle/valid/images'))
test_imgs = len(os.listdir('/kaggle/input/vechicle/test/images'))

print(f"\n✓ Train images: {train_imgs}")
print(f"✓ Valid images: {valid_imgs}")
print(f"✓ Test images: {test_imgs}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Show sample training image
img_path = '/kaggle/input/vechicle/train/images/' + os.listdir('/kaggle/input/vechicle/train/images')[0]
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title("Sample Training Image - Bangladesh Vehicle")
plt.axis('off')
plt.show()

In [ ]:
# Install Ultralytics YOLOv8
!pip install ultralytics -q

from ultralytics import YOLO
print("✓ YOLOv8 installed successfully!")

In [ ]:
# Create data.yaml
config = """
train: /kaggle/input/vechicle/train/images
val: /kaggle/input/vechicle/valid/images
test: /kaggle/input/vechicle/test/images

nc: 3
names: ['car', 'cng', 'bus']
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(config)

print("✓ Config created")
!cat /kaggle/working/data.yaml

In [ ]:
# Load pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    name='bangladesh_vehicle',
    patience=10
)

print("✓ Training complete!")

In [ ]:
from IPython.display import Image, display

base = '/kaggle/working/runs/detect/bangladesh_vehicle2/'

display(Image(base + 'results.png'))
display(Image(base + 'confusion_matrix.png'))
display(Image(base + 'BoxP_curve.png'))
display(Image(base + 'train_batch0.jpg'))


In [ ]:
import pandas as pd

results_df = pd.read_csv(
    '/kaggle/working/runs/detect/bangladesh_vehicle2/results.csv'
)

print("Last 5 Epochs:")
print(results_df.tail())

print("\n=== Best Performance ===")
print(f"Best mAP50: {results_df['metrics/mAP50(B)'].max():.3f}")
print(f"Best mAP50-95: {results_df['metrics/mAP50-95(B)'].max():.3f}")


In [ ]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/detect/bangladesh_vehicle2/weights/best.pt')

results = model.predict(
    source='/kaggle/input/vechicle/test/images/',
    save=True,
    conf=0.5,
    project='/kaggle/working/runs/detect',
    name='test_results'
)

print(f"✓ Tested on {len(results)} images")


In [ ]:
import glob
import matplotlib.pyplot as plt

# Get result images
result_imgs = glob.glob('/kaggle/working/runs/detect/test_results/*.jpg')

# Show first 6 detections
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, img_path in enumerate(result_imgs[:6]):
    ax = axes[i//3, i%3]
    img = plt.imread(img_path)
    ax.imshow(img)
    ax.set_title(f'Detection {i+1}', fontsize=14, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"✓ Total test images processed: {len(result_imgs)}")

In [ ]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/detect/bangladesh_vehicle2/weights/best.pt')

metrics = model.val(data='/kaggle/working/data.yaml')

print("\n=== Overall Performance ===")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

print("\n=== Per Class Performance ===")
print(f"Car mAP50: {metrics.box.maps[0]:.3f}")
print(f"CNG mAP50: {metrics.box.maps[1]:.3f}")
print(f"Bus mAP50: {metrics.box.maps[2]:.3f}")


In [ ]:
# Create final output folder
!mkdir -p /kaggle/working/final_output

# Copy trained model
!cp /kaggle/working/runs/detect/bangladesh_vehicle2/weights/best.pt \
    /kaggle/working/final_output/best_model.pt

# Copy training results
!cp /kaggle/working/runs/detect/bangladesh_vehicle2/results.png \
    /kaggle/working/final_output/

!cp /kaggle/working/runs/detect/bangladesh_vehicle2/confusion_matrix.png \
    /kaggle/working/final_output/

!cp /kaggle/working/runs/detect/bangladesh_vehicle2/results.csv \
    /kaggle/working/final_output/

# Copy test results
!mkdir -p /kaggle/working/final_output/test_samples
!cp /kaggle/working/runs/detect/test_results/*.jpg \
    /kaggle/working/final_output/test_samples/

# Verify
!ls -lh /kaggle/working/final_output/


In [ ]:
#Assignment

from ultralytics import YOLO
model = YOLO('/kaggle/working/runs/detect/bangladesh_vehicle2/weights/best.pt')
results = model.predict(
    source='/kaggle/input/traffic-f',
    save=True,
    conf=0.5,
    project='/kaggle/working/runs/detect',
    name='video_results'
)


print("✓ Video detection complete!")
print("✓ Video saved to: /kaggle/working/runs/detect/video_results/")

In [ ]:
!ls /kaggle/working/runs/detect/video_results/


In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    name='bangladesh_vehicle'
)
print("✓ Video detection complete!")
print("✓ Video saved to: /kaggle/working/runs/detect/video_results/")

In [ ]:
!pip install ultralytics -q


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/best.pt')
results = model.predict(
    source='/content/Heavy traffic is at the street of Dhaka Bangladesh..mp4',
    save=True,
    conf=0.5,
    project='/content/runs/detect',
    name='video_results'
)


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/best.pt')
results = model.predict(
    source='/content/Traffic Signal In Dhaka City (part-2) - Transport World (720p, h264).mp4',
    save=True,
    conf=0.5,
    project='/content/runs/detect',
    name='video_results'
)


In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('/content/best.pt')

results = model.predict(
    source='/content/Traffic Signal In Dhaka City (part-2) - Transport World (720p, h264).mp4',
    save=True,
    conf=0.5,
    project='/content/runs/detect',
    name='video_results',
    vid_stride=1
)
